# Notebook 04 — Inference Application
## Landmark Classification CNN · UCB MSc Data Science & AI

> **Business question:** Can the deployed model correctly identify landmarks in real-world images not seen during training?

**Phase 4 rubric targets:**
- predict_landmarks(img_path, k) using TorchScript
- Test on >=4 personal images not from the dataset
- Written analysis of strengths and weaknesses

> Run this notebook locally on PyCharm — inference is CPU-friendly.

In [ ]:
# ── Cell 0A: Mount Google Drive (Colab only) ───────────────
# Why: dataset lives in Google Drive. Must mount before any
# src.config import — TRAIN_DIR and TEST_DIR resolve to Drive paths.
# Skip this cell if running locally in PyCharm.
import os
if os.path.exists('/content'):
    from google.colab import drive
    drive.mount('/content/drive')
    import subprocess
    if not os.path.exists('/content/landmark-classifier'):
        subprocess.run(['git', 'clone',
            'https://github.com/guillermocarvajalvaca-dev/landmark-classifier.git',
            '/content/landmark-classifier'], check=True)
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'plotnine'], check=True)
    print('Colab environment ready')
else:
    print('Local environment detected')


In [ ]:
# ── Cell 1: Environment setup ──────────────────────────────
# Why first cell is always config: single source of truth for
# paths and hyperparameters. Any change propagates to all cells.
import logging
import os
import sys
from pathlib import Path

# Why explicit Colab path: Path('..').resolve() returns /content
# in Colab instead of the project root, breaking src.* imports.
PROJECT_ROOT = (
    Path('/content/landmark-classifier')
    if os.path.exists('/content/landmark-classifier/src')
    else Path('..').resolve()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(level=logging.INFO)
from src.config import EXTERNAL_DIR, MODELS_DIR, SEED
from src.utils import set_seed

set_seed(SEED)
scripted_models = sorted(MODELS_DIR.glob('*_scripted.pt'))
print('Available TorchScript models:')
for m in scripted_models:
    print(f'  {m.name}')

In [ ]:
# ── Cell 2: Select best TorchScript model ───────────────────────
# Why TorchScript: self-contained computation graph — no model.py
# dependency. Deployable with only torch installed.
finetune_models = [m for m in scripted_models if 'finetune' in m.name]
BEST_MODEL_PATH = finetune_models[-1] if finetune_models else scripted_models[-1]
print(f'Using model: {BEST_MODEL_PATH.name}')

In [ ]:
# ── Cell 3: Verify external images ─────────────────────────────
# Why verify before inference: a missing image raises a cryptic
# CUDA error mid-batch. Explicit check gives a clear message.
images = (
    list(EXTERNAL_DIR.glob('*.jpg'))
    + list(EXTERNAL_DIR.glob('*.jpeg'))
    + list(EXTERNAL_DIR.glob('*.png'))
    + list(EXTERNAL_DIR.glob('*.webp'))
)
print(f'External images found: {len(images)}')
for img in images:
    print(f'  {img.name}')
assert len(images) >= 4, f'Rubric requires >=4 images. Found: {len(images)}'

In [ ]:
# ── Cell 4: Inference on all external images ────────────────────
# Why predict_and_display: renders UCB palette bar chart alongside
# the image — communicates confidence, not just the top-1 label.
from src.predictor import predict_and_display

all_predictions = {}
for img_path in images:
    print(f'\n=== {img_path.name} ===')
    preds = predict_and_display(
        img_path            = img_path,
        k                   = 3,
        scripted_model_path = BEST_MODEL_PATH,
    )
    all_predictions[img_path.name] = preds
    for rank, (name, prob) in enumerate(preds, 1):
        print(f'  {rank}. {name.replace("_", " ")}: {prob:.2%}')

In [ ]:
# ── Cell 5: Prediction summary table ───────────────────────────
import pandas as pd

rows = []
for img_name, preds in all_predictions.items():
    for rank, (name, prob) in enumerate(preds, 1):
        rows.append({
            'Image'      : img_name,
            'Rank'       : rank,
            'Prediction' : name.replace('_', ' '),
            'Confidence' : f'{prob:.2%}',
        })
print(pd.DataFrame(rows).to_string(index=False))

## Analysis — Strengths and Weaknesses

*(Fill based on prediction results)*

### Strengths
- High confidence on iconic visually distinctive landmarks
- Top-3 includes correct class even when Top-1 is wrong

### Weaknesses
- Low confidence on underrepresented landmarks
- Confuses architecturally similar structures

## Final Deliverables Checklist

| Deliverable | Status |
|---|---|
| GitHub repo public (no dataset) | ⬜ |
| All 4 notebooks executed with outputs | ⬜ |
| README.md with YouTube link | ⬜ |
| Video 3-5 min uploaded | ⬜ |
| docs/informe_landmarks.pdf | ⬜ |
| git tag v1.0-entrega pushed | ⬜ |